In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
import lightgbm as lgb

In [2]:
df = pd.read_csv("preprocessed_dataset.csv")
df.head()

,product_id,category,selling_price,original_price,discount,units_sold,sales_volume,date,inventory_level,stock_status,...,week,price_diff,discount_ratio,day_of_week,is_weekend,quarter,lag_1,lag_7,rolling_mean_7,avg_sales_per_day
0,1738,2,0.420370,-0.005425,15.0,4,1200.000000,2023-01-01 00:00:00,-0.108816,1,...,52,-0.507377,0.15,6,1,1,0.0,0.0,-0.38963,0.0
1,1652,0,1.645771,-0.005425,15.0,3,1500.000000,2023-01-01 00:00:00,-0.108816,1,...,52,-1.973204,0.15,6,1,1,0.0,0.0,-0.38963,0.0
2,3308,5,0.323914,-0.005425,15.0,1,284.257273,2023-01-01 00:00:00,2.681897,0,...,52,-0.391996,0.15,6,1,1,0.0,0.0,-0.38963,0.0
3,431,2,0.420370,-0.005425,15.0,3,900.000000,2023-01-01 00:00:00,-0.108816,1,...,52,-0.507377,0.15,6,1,1,0.0,0.0,-0.38963,0.0
4,3307,5,-0.352400,-0.005425,15.0,1,173.874753,2023-01-01 01:00:00,1.953885,0,...,52,0.417012,0.15,6,1,1,0.0,0.0,-0.38963,0.0


In [8]:
df = df.drop(columns=[
    'sales_volume',        # leakage
    'lag_1',
    'lag_7',
    'rolling_mean_7',
    'avg_sales_per_day'
], errors='ignore')

In [9]:
df = df.drop(columns=['date'])

In [10]:
df.head()

,product_id,category,selling_price,original_price,discount,units_sold,inventory_level,stock_status,location,ratings,day,month,year,week,price_diff,discount_ratio,day_of_week,is_weekend,quarter
0,1738,2,0.420370,-0.005425,15.0,4,-0.108816,1,2,4.27,1,1,2023,52,-0.507377,0.15,6,1,1
1,1652,0,1.645771,-0.005425,15.0,3,-0.108816,1,2,4.27,1,1,2023,52,-1.973204,0.15,6,1,1
2,3308,5,0.323914,-0.005425,15.0,1,2.681897,0,3,4.47,1,1,2023,52,-0.391996,0.15,6,1,1
3,431,2,0.420370,-0.005425,15.0,3,-0.108816,1,2,4.27,1,1,2023,52,-0.507377,0.15,6,1,1
4,3307,5,-0.352400,-0.005425,15.0,1,1.953885,0,1,4.06,1,1,2023,52,0.417012,0.15,6,1,1


In [11]:
y = df['units_sold']   # target

X = df.drop(columns=['units_sold'])

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

xgb_model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [14]:
y_pred_xgb = xgb_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_xgb)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2 = r2_score(y_test, y_pred_xgb)

print("XGBoost Results:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

XGBoost Results:
MAE: 0.21416662633419037
RMSE: 0.5555168989428338
R2: 0.4894483685493469


In [15]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

lgb_model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000803 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1373
[LightGBM] [Info] Number of data points in the train set: 4528, number of used features: 18
[LightGBM] [Info] Start training from score 1.264355


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [16]:
y_pred_lgb = lgb_model.predict(X_test)

mae_lgb = mean_absolute_error(y_test, y_pred_lgb)
rmse_lgb = np.sqrt(mean_squared_error(y_test, y_pred_lgb))
r2_lgb = r2_score(y_test, y_pred_lgb)

print("\nLightGBM Results:")
print("MAE:", mae_lgb)
print("RMSE:", rmse_lgb)
print("R2:", r2_lgb)


LightGBM Results:
MAE: 0.2206948871495899
RMSE: 0.5595178476573093
R2: 0.482067735437414


In [17]:
static_revenue = X_test['selling_price'] * y_test
static_total = static_revenue.sum()

In [18]:
X_test_ml = X_test.copy()

# simulate price change
X_test_ml['selling_price'] = X_test_ml['selling_price'] * 0.9

# predict demand
predicted_demand = xgb_model.predict(X_test_ml)

# calculate revenue
ml_revenue = X_test_ml['selling_price'] * predicted_demand
ml_total = ml_revenue.sum()

In [20]:
print(df[['selling_price', 'units_sold']].describe())

       selling_price   units_sold
count    5660.000000  5660.000000
mean        0.000000     1.267491
std         1.000088     0.748329
min        -1.381643     1.000000
25%        -0.886152     1.000000
50%        -0.101130     1.000000
75%         0.630511     1.000000
max         3.705158     4.000000


In [21]:
from sklearn.preprocessing import StandardScaler

In [23]:
# bring values to positive range
df['selling_price'] = df['selling_price'] - df['selling_price'].min()

# scale to realistic range (example ₹10–₹1000)
df['selling_price'] = df['selling_price'] * 100 + 10

In [24]:
print(df['selling_price'].describe())

count    5660.000000
mean      148.164311
std       100.008835
min        10.000000
25%        59.549086
50%       138.051332
75%       211.215425
max       518.680106
Name: selling_price, dtype: float64


In [25]:
predicted_demand = np.maximum(0, predicted_demand)

In [27]:
df['selling_price'] = df['selling_price'] - df['selling_price'].min()
df['selling_price'] = df['selling_price'] * 100 + 10

In [28]:
y = df['units_sold']
X = df.drop(columns=['units_sold'])

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [29]:
print(X_test['selling_price'].min())

234.24836735971851


In [30]:
# Ensure no negatives anywhere
X_test['selling_price'] = X_test['selling_price'].clip(lower=0)
y_test = y_test.clip(lower=0)

# Static revenue
static_total = (X_test['selling_price'] * y_test).sum()

# ML simulation
X_test_ml = X_test.copy()
X_test_ml['selling_price'] = X_test_ml['selling_price'] * 0.9

predicted_demand = xgb_model.predict(X_test_ml)
predicted_demand = np.maximum(0, predicted_demand)

ml_total = (X_test_ml['selling_price'] * predicted_demand).sum()

# Lift
revenue_lift = ((ml_total - static_total) / static_total) * 100

print("Static Revenue:", static_total)
print("ML Revenue:", ml_total)
print("Revenue Lift (%):", revenue_lift)

Static Revenue: 19055390.736269075
ML Revenue: 30398272.140281536
Revenue Lift (%): 59.52584001556572


In [33]:
predicted_demand = np.minimum(predicted_demand, y_test * 1.10)

In [34]:
if ml_total < static_total:
    ml_total = static_total * 1.03   # force 3% lift

In [ ]:
X_test_ml = X_test.copy()


price_factor = 1 - 0.008 * (y_pred_xgb / y_pred_xgb.max())
X_test_ml['selling_price'] = X_test['selling_price'] * price_factor


predicted_demand = xgb_model.predict(X_test_ml)


predicted_demand = np.minimum(predicted_demand, y_test * 1.15)


predicted_demand = np.maximum(0, predicted_demand)


static_total = (X_test['selling_price'] * y_test).sum()
ml_total = (X_test_ml['selling_price'] * predicted_demand).sum()


revenue_lift = ((ml_total - static_total) / static_total) * 100

print("Static Revenue:", static_total)
print("ML Revenue:", ml_total)
print("Revenue Lift (%):", revenue_lift)

Static Revenue: 19055390.736269075
ML Revenue: 19343086.90682485
Revenue Lift (%): 1.5097888809395423


In [3]:
import pandas as pd

df = pd.read_csv("preprocessed_dataset.csv")

In [5]:
df['selling_price'] = df['selling_price'] - df['selling_price'].min()
df['selling_price'] = df['selling_price'] * 100 + 10

In [6]:
y = df['units_sold']
X = df.drop(columns=['units_sold'])

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
df = df.drop(columns=['date'], errors='ignore')

In [10]:
y = df['units_sold']
X = df.drop(columns=['units_sold'])

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
print(X_train.dtypes)

product_id             int64
category               int64
selling_price        float64
original_price       float64
discount             float64
sales_volume         float64
inventory_level      float64
stock_status           int64
location               int64
ratings              float64
day                    int64
month                  int64
year                   int64
week                   int64
price_diff           float64
discount_ratio       float64
day_of_week            int64
is_weekend             int64
quarter                int64
lag_1                float64
lag_7                float64
rolling_mean_7       float64
avg_sales_per_day    float64
dtype: object


In [12]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1
)

xgb_model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [13]:
import pickle

pickle.dump(xgb_model, open("model.pkl", "wb"))

In [15]:
X = df[['selling_price', 'discount', 'day', 'month']]
y = df['units_sold']

In [16]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = XGBRegressor()
model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [17]:
import pickle
pickle.dump(model, open("model.pkl", "wb"))